# Balkan Cinema - Data Collection & Integration

## Phase 1: Building the Master Balkan Movie Dataset

This notebook collects data from multiple sources and merges them into a single clean dataset.

**Data Sources:**
1. **Wikidata** (SPARQL) — our foundation: all films with Balkan country of origin
2. **IMDb Non-Commercial Datasets** — ratings, genres, runtime, cast/crew
3. **TMDb (Kaggle)** — budget, revenue, production companies, plot overviews
4. **Lumiere** — European cinema admissions (post-1996)

**Output:** `data/balkan_movies_master.csv` — one row per movie with all enriched fields

## 0. Setup & Configuration

In [ ]:
# Install dependencies (run once)
# !pip install pandas numpy requests beautifulsoup4 tqdm

In [4]:
import pandas as pd
import numpy as np
import requests
import json
import os
import time
import gzip
import csv
from io import StringIO
from bs4 import BeautifulSoup
from tqdm import tqdm

# Create data directory if it doesn't exist
os.makedirs('data', exist_ok=True)
os.makedirs('images', exist_ok=True)

# Define Balkan countries and their codes
BALKAN_COUNTRIES = {
    # Wikidata QIDs
    'wikidata': {
        'Q36704': 'Yugoslavia',
        'Q403':   'Serbia',
        'Q224':   'Croatia',
        'Q225':   'Bosnia and Herzegovina',
        'Q215':   'Slovenia',
        'Q236':   'Montenegro',
        'Q221':   'North Macedonia',
        'Q222':   'Albania',
        'Q41':    'Greece',
        'Q219':   'Bulgaria',
        'Q218':   'Romania',
        'Q1246':  'Kosovo',
    },
    # IMDb region codes (used in title.akas.tsv)
    'imdb_regions': ['RS', 'HR', 'BA', 'SI', 'ME', 'MK', 'AL', 'GR', 'BG', 'RO', 'XK', 'YU', 'CS'],
    # ISO country names for TMDb matching
    'tmdb_names': [
        'Yugoslavia', 'Serbia', 'Serbia and Montenegro', 'Croatia',
        'Bosnia and Herzegovina', 'Bosnia & Herzegovina', 'Slovenia', 'Montenegro',
        'North Macedonia', 'Macedonia', 'Albania', 'Greece', 'Bulgaria',
        'Romania', 'Kosovo', 'Federal Republic of Yugoslavia',
        'Socialist Federal Republic of Yugoslavia'
    ]
}

print(f"Tracking {len(BALKAN_COUNTRIES['wikidata'])} Balkan countries/entities")

Tracking 12 Balkan countries/entities


---
## 1. Wikidata — Foundation Dataset

We query Wikidata for ALL films with a Balkan country of origin. This gives us the most complete list of Balkan movies, including obscure titles not in IMDb or TMDb.

Wikidata provides: title, year, country, director, genre, IMDb ID (for cross-referencing), budget, box office, awards.

In [9]:
import re

def query_wikidata_balkan_films():
    """
    Query Wikidata for all films originating from Balkan countries.
    Returns a DataFrame with one row per film.
    
    NOTE: This query may take 1-3 minutes to run due to the volume of data.
    
    We use requests directly instead of SPARQLWrapper.convert() because
    some Wikidata labels contain control characters (newlines, tabs) inside
    JSON string values, which causes json.loads() to crash in strict mode.
    Using json.loads(text, strict=False) fixes this.
    """
    WIKIDATA_ENDPOINT = "https://query.wikidata.org/sparql"
    
    all_results = []
    
    for qid, country_name in BALKAN_COUNTRIES['wikidata'].items():
        print(f"  Querying Wikidata for: {country_name} ({qid})...")
        
        query = f"""
        SELECT DISTINCT
            ?film ?filmLabel ?imdb_id ?date ?countryLabel
            ?directorLabel ?genreLabel ?runtime
            ?box_office ?budget ?award_label
        WHERE {{
            ?film wdt:P31 wd:Q11424 .          # instance of: film
            ?film wdt:P495 wd:{qid} .           # country of origin
            
            OPTIONAL {{ ?film wdt:P345 ?imdb_id . }}        # IMDb ID
            OPTIONAL {{ ?film wdt:P577 ?date . }}            # publication date
            OPTIONAL {{ ?film wdt:P57 ?director . }}         # director
            OPTIONAL {{ ?film wdt:P136 ?genre . }}           # genre
            OPTIONAL {{ ?film wdt:P2047 ?runtime . }}        # duration
            OPTIONAL {{ ?film wdt:P2142 ?box_office . }}     # box office
            OPTIONAL {{ ?film wdt:P2130 ?budget . }}         # cost/budget
            OPTIONAL {{ ?film wdt:P166 ?award .
                       ?award rdfs:label ?award_label .
                       FILTER(LANG(?award_label) = \"en\") }}
            
            ?film wdt:P495 ?country .
            
            SERVICE wikibase:label {{ bd:serviceParam wikibase:language \"en,sr,hr,sl,mk,sq,el,bg,ro\". }}
        }}
        """
        
        try:
            # Use requests directly so we can control JSON parsing
            response = requests.get(
                WIKIDATA_ENDPOINT,
                params={'query': query, 'format': 'json'},
                headers={'User-Agent': 'BalkanCinemaResearch/1.0 (university project)'},
                timeout=120
            )
            response.raise_for_status()
            
            # KEY FIX: strict=False allows control characters (\n, \t, etc.)
            # inside JSON string values, which some Wikidata labels contain
            raw_text = response.text
            # Also strip any actual control chars from inside string values
            raw_text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f]', ' ', raw_text)
            results = json.loads(raw_text, strict=False)
            
            bindings = results['results']['bindings']
            
            for b in bindings:
                row = {
                    'wikidata_id': b.get('film', {}).get('value', '').split('/')[-1],
                    'title': b.get('filmLabel', {}).get('value', ''),
                    'imdb_id': b.get('imdb_id', {}).get('value', ''),
                    'year': b.get('date', {}).get('value', '')[:4] if b.get('date') else '',
                    'country': b.get('countryLabel', {}).get('value', country_name),
                    'director': b.get('directorLabel', {}).get('value', ''),
                    'genre_wikidata': b.get('genreLabel', {}).get('value', ''),
                    'runtime_wikidata': b.get('runtime', {}).get('value', ''),
                    'box_office_wikidata': b.get('box_office', {}).get('value', ''),
                    'budget_wikidata': b.get('budget', {}).get('value', ''),
                    'award': b.get('award_label', {}).get('value', ''),
                }
                all_results.append(row)
            
            print(f"    → Found {len(bindings)} rows for {country_name}")
            time.sleep(2)  # Be polite to the Wikidata API
            
        except requests.exceptions.HTTPError as e:
            print(f"    HTTP ERROR for {country_name}: {e}")
            if response.status_code == 429:
                print("    Rate limited — waiting 30 seconds...")
                time.sleep(30)
            else:
                time.sleep(5)
        except Exception as e:
            print(f"    ERROR for {country_name}: {e}")
            time.sleep(5)
    
    df = pd.DataFrame(all_results)
    print(f"\nTotal raw rows from Wikidata: {len(df)}")
    return df

# Run the query
print("=" * 60)
print("STEP 1: Querying Wikidata for Balkan films...")
print("=" * 60)
df_wikidata_raw = query_wikidata_balkan_films()

STEP 1: Querying Wikidata for Balkan films...
  Querying Wikidata for: Yugoslavia (Q36704)...
    → Found 2684 rows for Yugoslavia
  Querying Wikidata for: Serbia (Q403)...
    → Found 1718 rows for Serbia
  Querying Wikidata for: Croatia (Q224)...
    → Found 2555 rows for Croatia
  Querying Wikidata for: Bosnia and Herzegovina (Q225)...
    → Found 1774 rows for Bosnia and Herzegovina
  Querying Wikidata for: Slovenia (Q215)...
    → Found 4116 rows for Slovenia
  Querying Wikidata for: Montenegro (Q236)...
    → Found 214 rows for Montenegro
  Querying Wikidata for: North Macedonia (Q221)...
    → Found 1049 rows for North Macedonia
  Querying Wikidata for: Albania (Q222)...
    → Found 929 rows for Albania
  Querying Wikidata for: Greece (Q41)...
    → Found 6620 rows for Greece
  Querying Wikidata for: Bulgaria (Q219)...
    → Found 2226 rows for Bulgaria
  Querying Wikidata for: Romania (Q218)...
    → Found 5753 rows for Romania
  Querying Wikidata for: Kosovo (Q1246)...
    → F

In [10]:
def aggregate_wikidata(df):
    """
    Wikidata returns multiple rows per film (one per genre, director, award, etc.).
    We aggregate these into a single row per film.
    """
    if df.empty:
        return df
    
    # Group by wikidata_id and aggregate list-like fields
    agg_funcs = {
        'title': 'first',
        'imdb_id': 'first',
        'year': 'first',
        'country': lambda x: '|'.join(sorted(set(x.dropna().astype(str)))),
        'director': lambda x: '|'.join(sorted(set(x.dropna().astype(str)) - {''})),
        'genre_wikidata': lambda x: '|'.join(sorted(set(x.dropna().astype(str)) - {''})),
        'runtime_wikidata': 'first',
        'box_office_wikidata': 'first',
        'budget_wikidata': 'first',
        'award': lambda x: '|'.join(sorted(set(x.dropna().astype(str)) - {''})),
    }
    
    df_agg = df.groupby('wikidata_id').agg(agg_funcs).reset_index()
    
    # Clean up year
    df_agg['year'] = pd.to_numeric(df_agg['year'], errors='coerce')
    
    # Clean up runtime
    df_agg['runtime_wikidata'] = pd.to_numeric(df_agg['runtime_wikidata'], errors='coerce')
    
    # Clean up budget and box office
    df_agg['budget_wikidata'] = pd.to_numeric(df_agg['budget_wikidata'], errors='coerce')
    df_agg['box_office_wikidata'] = pd.to_numeric(df_agg['box_office_wikidata'], errors='coerce')
    
    # Count awards
    df_agg['num_awards'] = df_agg['award'].apply(lambda x: len(x.split('|')) if x else 0)
    
    print(f"Aggregated to {len(df_agg)} unique films")
    print(f"  - Films with IMDb ID: {df_agg['imdb_id'].notna().sum() & (df_agg['imdb_id'] != '').sum()}")
    print(f"  - Year range: {df_agg['year'].min():.0f} - {df_agg['year'].max():.0f}")
    print(f"  - Countries: {df_agg['country'].nunique()}")
    
    return df_agg

df_wikidata = aggregate_wikidata(df_wikidata_raw)
df_wikidata.head()

Aggregated to 6235 unique films
  - Films with IMDb ID: 4120
  - Year range: 1898 - 2026
  - Countries: 552


,wikidata_id,title,imdb_id,year,country,director,genre_wikidata,runtime_wikidata,box_office_wikidata,budget_wikidata,award,num_awards
0,Q1001102,The Rat Savior,tt0074701,1976.0,Socialist Federal Republic of Yugoslavia|Yugos...,Krsto Papić,horror film|science fiction film,75.0,NaN,NaN,,0
1,Q100255627,Forgotten Christmas,tt9316574,2021.0,Norway|Romania,Andrea Eckerbom,Christmas film|film based on literature,70.0,NaN,28200000.0,,0
2,Q100535001,Cat in the Wall,tt8471116,2019.0,Bulgaria|France|United Kingdom,Mina Mileva|Vesela Kazakova,fiction film,92.0,NaN,NaN,,0
3,Q100564447,Motherland,tt10805618,2019.0,Germany|Greece|Latvia|Lithuania,Tomas Vengris,fiction film,97.0,NaN,NaN,,0
4,Q100718251,Full Moon,tt9299590,2019.0,Bosnia and Herzegovina,Nermin Hamzagić,thriller film,85.0,NaN,NaN,,0


In [ ]:
# Save intermediate result
df_wikidata.to_csv('data/wikidata_balkan_films.csv', index=False)
print(f"Saved {len(df_wikidata)} Wikidata films to data/wikidata_balkan_films.csv")

---
## 2. IMDb — Ratings, Genres, Runtime, Cast/Crew

Download the IMDb TSV files from https://datasets.imdbws.com/ and extract them into the `data/` folder:
- `title.basics.tsv`
- `title.ratings.tsv`
- `title.crew.tsv`
- `title.principals.tsv`
- `title.akas.tsv`
- `name.basics.tsv`

Some of these files are very large (title.akas is ~1.5GB, ~40M rows).
We use **chunked reading** to avoid memory crashes — the file is read in 500K-row
pieces and filtered immediately, so only the small Balkan subset stays in memory.

In [5]:
def load_imdb_basics(filepath='data/title.basics.tsv'):
    """
    Load IMDb title.basics - contains titleType, primaryTitle, year, genres, runtime.
    Filter to only movies (not TV episodes, shorts, etc.)
    Uses chunked reading to save memory — only keeps movie rows.
    """
    print("Loading IMDb title.basics (chunked, keeping only movies)...")
    chunks = []
    for chunk in pd.read_csv(filepath, sep='\t', low_memory=False, na_values='\\N', chunksize=500_000):
        filtered = chunk[chunk['titleType'].isin(['movie', 'tvMovie'])]
        if len(filtered) > 0:
            chunks.append(filtered)
    
    df = pd.concat(chunks, ignore_index=True)
    df['startYear'] = pd.to_numeric(df['startYear'], errors='coerce')
    
    print(f"  Loaded {len(df)} movies from IMDb basics")
    return df

def load_imdb_ratings(filepath='data/title.ratings.tsv'):
    """Load IMDb ratings - averageRating and numVotes. Small file, loads fully."""
    print("Loading IMDb title.ratings...")
    df = pd.read_csv(filepath, sep='\t', low_memory=False, na_values='\\N')
    print(f"  Loaded {len(df)} ratings")
    return df

def load_imdb_akas(filepath='data/title.akas.tsv'):
    """
    Load IMDb title.akas - contains region codes.
    This file is HUGE (~40M rows, 1.5GB+). We read it in chunks
    and only keep rows with Balkan region codes.
    Peak memory: ~200MB instead of ~6GB.
    """
    print("Loading IMDb title.akas in chunks (filtering to Balkan regions)...")
    balkan_regions = set(BALKAN_COUNTRIES['imdb_regions'])
    
    chunks = []
    rows_read = 0
    for chunk in pd.read_csv(filepath, sep='\t', low_memory=False, na_values='\\N', chunksize=500_000):
        rows_read += len(chunk)
        # Filter immediately — only keep Balkan region rows
        balkan_rows = chunk[chunk['region'].isin(balkan_regions)]
        if len(balkan_rows) > 0:
            chunks.append(balkan_rows)
        # Progress indicator
        print(f"    Processed {rows_read:,} rows, found {sum(len(c) for c in chunks):,} Balkan rows so far...", end='\r')
    
    print()  # newline after progress
    
    if chunks:
        df_balkan = pd.concat(chunks, ignore_index=True)
    else:
        df_balkan = pd.DataFrame()
    
    print(f"  Found {df_balkan['titleId'].nunique()} unique titles with Balkan region codes")
    print(f"  Total Balkan AKA rows: {len(df_balkan)}")
    return df_balkan

def load_imdb_crew(filepath='data/title.crew.tsv'):
    """Load IMDb crew data - directors and writers. Moderate size, loads fully."""
    print("Loading IMDb title.crew...")
    df = pd.read_csv(filepath, sep='\t', low_memory=False, na_values='\\N')
    print(f"  Loaded {len(df)} crew entries")
    return df

def load_imdb_principals(filepath='data/title.principals.tsv'):
    """
    Load IMDb principals - top-billed cast and crew per movie.
    Also a large file — we only load it if needed later.
    For now we skip it in the main pipeline to save memory.
    """
    print("Loading IMDb title.principals (chunked)...")
    chunks = []
    for chunk in pd.read_csv(filepath, sep='\t', low_memory=False, na_values='\\N', chunksize=500_000):
        chunks.append(chunk)
    df = pd.concat(chunks, ignore_index=True)
    print(f"  Loaded {len(df)} principal entries")
    return df

def load_imdb_names(filepath='data/name.basics.tsv'):
    """
    Load IMDb name.basics - to resolve person IDs to names.
    We only load the columns we need to save memory.
    """
    print("Loading IMDb name.basics (selected columns only)...")
    df = pd.read_csv(filepath, sep='\t', low_memory=False, na_values='\\N',
                     usecols=['nconst', 'primaryName', 'birthYear', 'deathYear'])
    print(f"  Loaded {len(df)} names")
    return df

In [6]:
# Load all IMDb data
print("=" * 60)
print("STEP 2: Loading IMDb datasets...")
print("=" * 60)
print("Make sure you've downloaded the TSV files to the data/ folder!")
print("Download from: https://datasets.imdbws.com/")
print()

imdb_basics = load_imdb_basics()
imdb_ratings = load_imdb_ratings()
imdb_akas = load_imdb_akas()
imdb_crew = load_imdb_crew()
imdb_names = load_imdb_names()
# imdb_pricipals = load_imdb_principals()

STEP 2: Loading IMDb datasets...
Make sure you've downloaded the TSV files to the data/ folder!
Download from: https://datasets.imdbws.com/

Loading IMDb title.basics (chunked, keeping only movies)...
  Loaded 895799 movies from IMDb basics
Loading IMDb title.ratings...
  Loaded 1652901 ratings
Loading IMDb title.akas in chunks (filtering to Balkan regions)...
    Processed 55,532,706 rows, found 228,535 Balkan rows so far...
  Found 133888 unique titles with Balkan region codes
  Total Balkan AKA rows: 228535
Loading IMDb title.crew...
  Loaded 12388768 crew entries
Loading IMDb name.basics (selected columns only)...
  Loaded 15190299 names


In [11]:
def build_imdb_balkan_set(df_wikidata, imdb_basics, imdb_ratings, imdb_akas, imdb_crew, imdb_names):
    """
    Build the set of Balkan IMDb IDs from two sources:
    1. IMDb IDs found in our Wikidata results
    2. IMDb titles that have Balkan region codes in title.akas
    
    Then enrich with ratings, genres, crew info.
    """
    # --- Collect all Balkan IMDb IDs ---
    # Source 1: From Wikidata
    wikidata_imdb_ids = set(df_wikidata[df_wikidata['imdb_id'] != '']['imdb_id'].dropna().unique())
    print(f"IMDb IDs from Wikidata: {len(wikidata_imdb_ids)}")
    
    # Source 2: From IMDb AKAs (films with Balkan region codes)
    akas_imdb_ids = set(imdb_akas['titleId'].unique())
    print(f"IMDb IDs from AKAs (Balkan regions): {len(akas_imdb_ids)}")
    
    # Combine
    all_balkan_ids = wikidata_imdb_ids | akas_imdb_ids
    print(f"Combined unique IMDb IDs: {len(all_balkan_ids)}")
    
    # --- Filter IMDb basics to our IDs ---
    df_imdb = imdb_basics[imdb_basics['tconst'].isin(all_balkan_ids)].copy()
    print(f"Matched in IMDb basics (movies only): {len(df_imdb)}")
    
    # --- Merge ratings ---
    df_imdb = df_imdb.merge(imdb_ratings, on='tconst', how='left')
    
    # --- Merge crew (directors) ---
    df_crew_filtered = imdb_crew[imdb_crew['tconst'].isin(all_balkan_ids)].copy()
    df_imdb = df_imdb.merge(df_crew_filtered[['tconst', 'directors', 'writers']], on='tconst', how='left')
    
    # --- Resolve director names ---
    def resolve_names(nconst_str, names_lookup):
        """Convert comma-separated nconst IDs to names."""
        if pd.isna(nconst_str):
            return ''
        ids = str(nconst_str).split(',')
        names = [names_lookup.get(nid, nid) for nid in ids]
        return '|'.join(names)
    
    names_lookup = dict(zip(imdb_names['nconst'], imdb_names['primaryName']))
    df_imdb['director_names'] = df_imdb['directors'].apply(lambda x: resolve_names(x, names_lookup))
    df_imdb['writer_names'] = df_imdb['writers'].apply(lambda x: resolve_names(x, names_lookup))
    
    # --- Add region info from AKAs ---
    region_per_title = imdb_akas.groupby('titleId')['region'].apply(
        lambda x: '|'.join(sorted(set(x.dropna())))
    ).reset_index()
    region_per_title.columns = ['tconst', 'balkan_regions']
    df_imdb = df_imdb.merge(region_per_title, on='tconst', how='left')
    
    # Rename columns for clarity
    df_imdb = df_imdb.rename(columns={
        'tconst': 'imdb_id',
        'primaryTitle': 'title_imdb',
        'originalTitle': 'original_title',
        'startYear': 'year_imdb',
        'runtimeMinutes': 'runtime_imdb',
        'genres': 'genres_imdb',
        'averageRating': 'imdb_rating',
        'numVotes': 'imdb_votes',
    })
    
    # Drop unnecessary columns
    df_imdb = df_imdb.drop(columns=['titleType', 'isAdult', 'endYear', 'directors', 'writers'], errors='ignore')
    
    print(f"\nFinal IMDb enriched dataset: {len(df_imdb)} films")
    print(f"  - With ratings: {df_imdb['imdb_rating'].notna().sum()}")
    print(f"  - With genres: {df_imdb['genres_imdb'].notna().sum()}")
    
    return df_imdb

df_imdb = build_imdb_balkan_set(df_wikidata, imdb_basics, imdb_ratings, imdb_akas, imdb_crew, imdb_names)
df_imdb.head()

IMDb IDs from Wikidata: 5176
IMDb IDs from AKAs (Balkan regions): 133888
Combined unique IMDb IDs: 134675
Matched in IMDb basics (movies only): 82321

Final IMDb enriched dataset: 82321 films
  - With ratings: 73488
  - With genres: 81131


,imdb_id,title_imdb,original_title,year_imdb,runtime_imdb,genres_imdb,imdb_rating,imdb_votes,director_names,writer_names,balkan_regions
0,tt0000574,The Story of the Kelly Gang,The Story of the Kelly Gang,1906.0,70.0,"Action,Adventure,Biography",6.0,1065.0,Charles Tait,Charles Tait,RS
1,tt0001475,Amor fatal,Amor fatal,1911.0,NaN,"Drama,Romance",7.6,26.0,Grigore Brezeanu,Grigore Brezeanu,RO
2,tt0002101,Cleopatra,Cleopatra,1912.0,100.0,"Drama,History",5.1,669.0,Charles L. Gaskill,Victorien Sardou|Charles L. Gaskill,GR
3,tt0002405,Oliver Twist,Oliver Twist,1912.0,NaN,Drama,5.1,48.0,,Charles Dickens,BG
4,tt0002406,Oliver Twist,Oliver Twist,1912.0,NaN,Drama,4.6,30.0,Thomas Bentley,Thomas Bentley|Charles Dickens,BG


In [12]:
# Save intermediate
df_imdb.to_csv('data/imdb_balkan_films.csv', index=False)
print(f"Saved {len(df_imdb)} IMDb-enriched Balkan films")

Saved 82321 IMDb-enriched Balkan films


---
## 3. TMDb (Kaggle) — Budget, Revenue, Plot Overviews

Download from Kaggle: https://www.kaggle.com/datasets/asaniczka/tmdb-movies-dataset-2023-930k-movies

Place the CSV file in `data/` as `tmdb_movies.csv`.

This dataset has columns like: id, title, release_date, budget, revenue, production_countries, 
genres, overview, vote_average, vote_count, imdb_id, etc.

In [13]:
def load_tmdb_balkan(filepath='data/tmdb_movies.csv'):
    """
    Load TMDb dataset and filter to Balkan productions.
    The TMDb dataset includes production_countries which lets us filter directly.
    """
    print("Loading TMDb dataset...")
    df = pd.read_csv(filepath, low_memory=False)
    print(f"  Total TMDb movies: {len(df)}")
    
    # Filter by production countries containing any Balkan country
    balkan_names = BALKAN_COUNTRIES['tmdb_names']
    
    # The production_countries column format varies by dataset version.
    # It might be JSON-like or pipe-separated. Handle both:
    def is_balkan(countries_str):
        if pd.isna(countries_str):
            return False
        countries_str = str(countries_str)
        return any(name.lower() in countries_str.lower() for name in balkan_names)
    
    df_balkan = df[df['production_countries'].apply(is_balkan)].copy()
    print(f"  Balkan films in TMDb: {len(df_balkan)}")
    
    # Keep useful columns
    cols_to_keep = [
        'id', 'title', 'original_title', 'release_date', 'budget', 'revenue',
        'production_countries', 'production_companies', 'genres', 'overview',
        'vote_average', 'vote_count', 'imdb_id', 'popularity', 'original_language',
        'spoken_languages', 'keywords', 'runtime'
    ]
    # Only keep columns that exist in the dataset
    cols_available = [c for c in cols_to_keep if c in df_balkan.columns]
    df_balkan = df_balkan[cols_available].copy()
    
    # Rename to avoid conflicts
    rename_map = {
        'id': 'tmdb_id',
        'title': 'title_tmdb',
        'budget': 'budget_tmdb',
        'revenue': 'revenue_tmdb',
        'vote_average': 'tmdb_rating',
        'vote_count': 'tmdb_votes',
        'genres': 'genres_tmdb',
        'runtime': 'runtime_tmdb',
        'overview': 'plot_overview',
    }
    df_balkan = df_balkan.rename(columns={k: v for k, v in rename_map.items() if k in df_balkan.columns})
    
    return df_balkan

print("=" * 60)
print("STEP 3: Loading TMDb dataset...")
print("=" * 60)
df_tmdb = load_tmdb_balkan()
df_tmdb.head()

STEP 3: Loading TMDb dataset...
Loading TMDb dataset...
  Total TMDb movies: 1392699
  Balkan films in TMDb: 15759


,tmdb_id,title_tmdb,original_title,release_date,budget_tmdb,revenue_tmdb,production_countries,production_companies,genres_tmdb,plot_overview,tmdb_rating,tmdb_votes,imdb_id,popularity,original_language,spoken_languages,keywords,runtime_tmdb
162,1271,300,300,2007-03-07,65000000,456082343,"Bulgaria, Canada, United States of America","Virtual Studios, Legendary Pictures, Hollywood...","Action, Adventure, War","Based on Frank Miller's graphic novel, ""300"" i...",7.175,12812,tt0416449,44.918,en,English,"epic, army, narration, blood splatter, gore, b...",117
484,27578,The Expendables,The Expendables,2010-08-03,80000000,274470394,"Bulgaria, United States of America","Nimar Studios, Rogue Marble, Millennium Media,...","Thriller, Adventure, Action",Barney Ross leads a band of highly skilled mer...,6.227,7225,tt1320253,89.721,en,"English, Spanish","rescue, sniper, island, martial arts, tattoo, ...",103
593,76163,The Expendables 2,The Expendables 2,2012-08-08,100000000,314975955,"Bulgaria, United States of America","Nu Image, Millennium Media, Lionsgate, Thunder...","Action, Adventure, Thriller",Mr. Church reunites the Expendables for what s...,6.323,6289,tt1764651,59.833,en,English,"airplane, loss of loved one, airplane crash, b...",103
654,2454,The Chronicles of Narnia: Prince Caspian,The Chronicles of Narnia: Prince Caspian,2008-05-15,225000000,419665568,"Czech Republic, Poland, Slovenia, United State...","Walt Disney Pictures, Walden Media, Stillking ...","Adventure, Family, Fantasy",One year after their incredible adventures in ...,6.616,5916,tt0499448,39.516,en,English,"witch, epic, sibling relationship, based on no...",150
683,390043,The Hitman's Bodyguard,The Hitman's Bodyguard,2017-08-16,30000000,176586701,"Bulgaria, Canada, France, Hong Kong, Netherlan...","Campbell Grobman Films, East Light Media, Nu B...","Action, Comedy","The world’s top bodyguard gets a new client, a...",6.891,5732,tt1959563,28.214,en,"French, English, Russian","assassin, court, england, hitman, bodyguard, t...",118


In [14]:
df_tmdb.to_csv('data/tmdb_balkan_films.csv', index=False)
print(f"Saved {len(df_tmdb)} TMDb Balkan films")

Saved 15759 TMDb Balkan films


---
## 4. Wikipedia Scraping — Film Lists by Country

As a supplement, we scrape Wikipedia's structured film lists.
These are useful for catching films not in Wikidata or IMDb.

In [18]:
def scrape_wikipedia_film_list(url, country_name):
    """
    Scrape a Wikipedia 'List of X films' page.
    These pages typically have tables with Title, Director, Year columns.
    """
    print(f"  Scraping: {country_name}...")
    try:
        response = requests.get(url, timeout=30)
        # Use lxml explicitly — html5lib 1.1 is broken on Python 3.14
        tables = pd.read_html(StringIO(response.text), flavor='lxml')
        
        # Find the largest table(s) — those are likely the film lists
        film_tables = [t for t in tables if len(t) > 5]
        
        if not film_tables:
            print(f"    No substantial tables found for {country_name}")
            return pd.DataFrame()
        
        # Combine all substantial tables
        combined = pd.concat(film_tables, ignore_index=True)
        combined['country_wiki'] = country_name
        
        print(f"    Found {len(combined)} entries")
        return combined
        
    except Exception as e:
        print(f"    ERROR: {e}")
        return pd.DataFrame()

# Wikipedia film list URLs
WIKI_FILM_LISTS = {
    'Yugoslav': 'https://en.wikipedia.org/wiki/List_of_Yugoslav_films',
    'Serbian': 'https://en.wikipedia.org/wiki/List_of_Serbian_films',
    'Croatian': 'https://en.wikipedia.org/wiki/List_of_Croatian_films',
    'Slovenian': 'https://en.wikipedia.org/wiki/List_of_Slovenian_films',
    'Bosnian': 'https://en.wikipedia.org/wiki/List_of_Bosnia_and_Herzegovina_films',
    'Macedonian': 'https://en.wikipedia.org/wiki/List_of_Macedonian_films',
    'Albanian': 'https://en.wikipedia.org/wiki/List_of_Albanian_films',
    'Greek': 'https://en.wikipedia.org/wiki/List_of_Greek_films',
    'Bulgarian': 'https://en.wikipedia.org/wiki/List_of_Bulgarian_films',
    'Romanian': 'https://en.wikipedia.org/wiki/List_of_Romanian_films',
    'Montenegrin': 'https://en.wikipedia.org/wiki/List_of_Montenegrin_films',
}

print("=" * 60)
print("STEP 4 (optional): Scraping Wikipedia film lists...")
print("=" * 60)

wiki_dfs = []
for country, url in WIKI_FILM_LISTS.items():
    df_wiki = scrape_wikipedia_film_list(url, country)
    if not df_wiki.empty:
        wiki_dfs.append(df_wiki)
    time.sleep(1)

if wiki_dfs:
    df_wikipedia = pd.concat(wiki_dfs, ignore_index=True)
    print(f"\nTotal Wikipedia entries: {len(df_wikipedia)}")
else:
    df_wikipedia = pd.DataFrame()
    print("No Wikipedia data collected (this is optional)")

STEP 4 (optional): Scraping Wikipedia film lists...
  Scraping: Yugoslav...
    ERROR: No tables found matching regex '.+'
  Scraping: Serbian...
    ERROR: No tables found matching regex '.+'
  Scraping: Croatian...
    ERROR: No tables found matching regex '.+'
  Scraping: Slovenian...
    ERROR: No tables found matching regex '.+'
  Scraping: Bosnian...
    ERROR: No tables found matching regex '.+'
  Scraping: Macedonian...
    ERROR: No tables found matching regex '.+'
  Scraping: Albanian...
    ERROR: No tables found matching regex '.+'
  Scraping: Greek...
    ERROR: No tables found matching regex '.+'
  Scraping: Bulgarian...
    ERROR: No tables found matching regex '.+'
  Scraping: Romanian...
    ERROR: No tables found matching regex '.+'
  Scraping: Montenegrin...
    ERROR: No tables found matching regex '.+'
No Wikipedia data collected (this is optional)


---
## 5. MERGE — Build the Master Dataset

Now we merge all sources. The strategy:
1. Start with **Wikidata** as the base (most comprehensive list of Balkan films)
2. **LEFT JOIN** IMDb data on `imdb_id` to add ratings, genres, runtime
3. **LEFT JOIN** TMDb data on `imdb_id` to add budget, revenue, plot overviews
4. **UNION** any films found in IMDb AKAs or Wikipedia that weren't in Wikidata

The result is one CSV file: `data/balkan_movies_master.csv`

In [19]:
def merge_all_sources(df_wikidata, df_imdb, df_tmdb):
    """
    Merge all data sources into a single master dataset.
    Priority: Wikidata as base → enrich with IMDb → enrich with TMDb.
    Then add IMDb-only films (from AKAs) that Wikidata missed.
    """
    print("=" * 60)
    print("STEP 5: Merging all data sources...")
    print("=" * 60)
    
    # ---- A) Start with Wikidata, merge IMDb ----
    # Only merge where Wikidata has a valid imdb_id
    df_master = df_wikidata.merge(
        df_imdb, on='imdb_id', how='left', suffixes=('', '_imdb_dup')
    )
    print(f"After Wikidata + IMDb merge: {len(df_master)} rows")
    
    # ---- B) Merge TMDb ----
    if 'imdb_id' in df_tmdb.columns:
        df_master = df_master.merge(
            df_tmdb, on='imdb_id', how='left', suffixes=('', '_tmdb_dup')
        )
        print(f"After TMDb merge: {len(df_master)} rows")
    
    # ---- C) Add IMDb-only films not in Wikidata ----
    wikidata_imdb_ids = set(df_wikidata['imdb_id'].dropna().unique())
    imdb_only = df_imdb[~df_imdb['imdb_id'].isin(wikidata_imdb_ids)].copy()
    
    if len(imdb_only) > 0:
        # Also merge TMDb into these
        if 'imdb_id' in df_tmdb.columns:
            imdb_only = imdb_only.merge(
                df_tmdb, on='imdb_id', how='left', suffixes=('', '_tmdb_dup')
            )
        
        # Append to master
        df_master = pd.concat([df_master, imdb_only], ignore_index=True, sort=False)
        print(f"Added {len(imdb_only)} IMDb-only films → Total: {len(df_master)}")
    
    # ---- D) Consolidate columns ----
    # Prefer Wikidata title, fall back to IMDb, then TMDb
    df_master['title_final'] = df_master['title'].fillna(
        df_master.get('title_imdb', pd.Series())
    ).fillna(
        df_master.get('title_tmdb', pd.Series())
    )
    
    # Consolidate year
    df_master['year_final'] = df_master['year'].fillna(
        df_master.get('year_imdb', pd.Series())
    )
    
    # Consolidate runtime (prefer IMDb)
    df_master['runtime_final'] = df_master.get('runtime_imdb', pd.Series()).fillna(
        df_master.get('runtime_tmdb', pd.Series())
    ).fillna(
        df_master.get('runtime_wikidata', pd.Series())
    )
    
    # Consolidate genres (prefer IMDb, supplement with Wikidata)
    df_master['genres_final'] = df_master.get('genres_imdb', pd.Series()).fillna(
        df_master.get('genres_tmdb', pd.Series())
    ).fillna(
        df_master.get('genre_wikidata', pd.Series())
    )
    
    # Consolidate budget (prefer TMDb, which has more financial data)
    df_master['budget_final'] = df_master.get('budget_tmdb', pd.Series()).fillna(
        df_master.get('budget_wikidata', pd.Series())
    )
    # Replace 0 budgets with NaN (TMDb uses 0 for unknown)
    df_master.loc[df_master['budget_final'] == 0, 'budget_final'] = np.nan
    
    # Revenue
    df_master['revenue_final'] = df_master.get('revenue_tmdb', pd.Series()).fillna(
        df_master.get('box_office_wikidata', pd.Series())
    )
    df_master.loc[df_master['revenue_final'] == 0, 'revenue_final'] = np.nan
    
    # ---- E) Create era/period column for analysis ----
    def assign_era(year):
        if pd.isna(year):
            return 'Unknown'
        year = int(year)
        if year < 1920:
            return '1. Pioneers (pre-1920)'
        elif year < 1945:
            return '2. Early Cinema (1920-1944)'
        elif year < 1963:
            return '3. Post-WWII / Socialist Realism (1945-1962)'
        elif year < 1975:
            return '4. Black Wave & New Film (1963-1974)'
        elif year < 1991:
            return '5. Late Yugoslavia (1975-1990)'
        elif year < 2001:
            return '6. Yugoslav Wars Era (1991-2000)'
        elif year < 2010:
            return '7. Rebuilding (2001-2009)'
        else:
            return '8. Contemporary (2010+)'
    
    df_master['era'] = df_master['year_final'].apply(assign_era)
    
    print(f"\n{'=' * 40}")
    print(f"MASTER DATASET: {len(df_master)} films")
    print(f"{'=' * 40}")
    
    return df_master

df_master = merge_all_sources(df_wikidata, df_imdb, df_tmdb)

STEP 5: Merging all data sources...
After Wikidata + IMDb merge: 6235 rows
After TMDb merge: 6236 rows
Added 77667 IMDb-only films → Total: 83903

MASTER DATASET: 83903 films


In [20]:
# ---- Select final columns for the clean master CSV ----
# We keep both the consolidated columns and the source-specific ones

final_columns = [
    # Identifiers
    'wikidata_id', 'imdb_id', 'tmdb_id',
    # Core info (consolidated)
    'title_final', 'original_title', 'year_final', 'era',
    'country', 'director', 'director_names',
    'genres_final', 'runtime_final',
    # Ratings
    'imdb_rating', 'imdb_votes', 'tmdb_rating', 'tmdb_votes',
    # Financial
    'budget_final', 'revenue_final',
    # Enrichment
    'plot_overview', 'production_companies', 'production_countries',
    'original_language', 'spoken_languages',
    'writer_names', 'award', 'num_awards',
    'balkan_regions',
]

# Only keep columns that exist
final_columns = [c for c in final_columns if c in df_master.columns]

df_clean = df_master[final_columns].copy()

# Remove exact duplicates
df_clean = df_clean.drop_duplicates(subset=['imdb_id'], keep='first')
# Also deduplicate by title+year for films without IMDb ID
mask_no_imdb = df_clean['imdb_id'].isna() | (df_clean['imdb_id'] == '')
df_with_imdb = df_clean[~mask_no_imdb]
df_without_imdb = df_clean[mask_no_imdb].drop_duplicates(
    subset=['title_final', 'year_final'], keep='first'
)
df_clean = pd.concat([df_with_imdb, df_without_imdb], ignore_index=True)

print(f"Final clean dataset: {len(df_clean)} unique films")
print(f"\nColumn overview:")
print(df_clean.dtypes)
print(f"\nMissing values:")
print(df_clean.isnull().sum())

Final clean dataset: 82844 unique films

Column overview:
wikidata_id                 str
imdb_id                     str
tmdb_id                 float64
title_final                 str
original_title              str
year_final              float64
era                         str
country                     str
director                    str
director_names              str
genres_final                str
runtime_final            object
imdb_rating             float64
imdb_votes              float64
tmdb_rating             float64
tmdb_votes              float64
budget_final            float64
revenue_final           float64
plot_overview               str
production_companies        str
production_countries        str
original_language           str
spoken_languages            str
writer_names                str
award                       str
num_awards              float64
balkan_regions              str
dtype: object

Missing values:
wikidata_id             77667
imdb_id          

In [21]:
# ---- Quick data quality check ----
print("DATASET OVERVIEW")
print("=" * 50)
print(f"Total films: {len(df_clean)}")
print(f"Year range: {df_clean['year_final'].min():.0f} - {df_clean['year_final'].max():.0f}")
print(f"\nFilms per era:")
print(df_clean['era'].value_counts().sort_index())
print(f"\nFilms per country (top 10):")
# Split pipe-separated countries and count
countries_exploded = df_clean['country'].dropna().str.split('|').explode()
print(countries_exploded.value_counts().head(10))
print(f"\nData completeness:")
for col in ['imdb_rating', 'budget_final', 'revenue_final', 'plot_overview', 'genres_final']:
    if col in df_clean.columns:
        pct = df_clean[col].notna().mean() * 100
        print(f"  {col}: {pct:.1f}% filled")

DATASET OVERVIEW
Total films: 82844
Year range: 1898 - 2031

Films per era:
era
1. Pioneers (pre-1920)                            139
2. Early Cinema (1920-1944)                      6183
3. Post-WWII / Socialist Realism (1945-1962)     8473
4. Black Wave & New Film (1963-1974)             8808
5. Late Yugoslavia (1975-1990)                  11016
6. Yugoslav Wars Era (1991-2000)                 9045
7. Rebuilding (2001-2009)                       12935
8. Contemporary (2010+)                         25588
Unknown                                           657
Name: count, dtype: int64

Films per country (top 10):
country
Romania       1386
Greece        1089
Yugoslavia    1006
Serbia         454
Bulgaria       423
Albania        335
Croatia        300
France         234
Germany        198
Slovenia       177
Name: count, dtype: int64

Data completeness:
  imdb_rating: 88.7% filled
  budget_final: 0.4% filled
  revenue_final: 0.2% filled
  plot_overview: 8.1% filled
  genres_final: 98.7%

In [22]:
# ---- SAVE THE MASTER DATASET ----
df_clean.to_csv('data/balkan_movies_master.csv', index=False, encoding='utf-8-sig')
print(f"\n*** SAVED: data/balkan_movies_master.csv ({len(df_clean)} films) ***")
print(f"File size: {os.path.getsize('data/balkan_movies_master.csv') / 1024:.0f} KB")
print("\nYou can now proceed to 02_analysis.ipynb for visualization and analysis!")


*** SAVED: data/balkan_movies_master.csv (82844 films) ***
File size: 17423 KB

You can now proceed to 02_analysis.ipynb for visualization and analysis!


---
## 6. Lumiere Admissions Data (Economic Analysis)

Lumiere data must be collected manually from https://lumiere.obs.coe.int/

**How to collect:**
1. Go to https://lumiere.obs.coe.int/search/
2. Search by producing country: select each Balkan country one at a time
3. Download the results as Excel (limited to 200 results per query)
4. Save each file in `data/lumiere/` folder

The cell below merges those files if you've collected them.

In [23]:
def load_lumiere_data(lumiere_dir='data/lumiere/'):
    """
    Load and merge all Lumiere Excel exports.
    Expected columns: Title, Directors, Producing countries, Year, Admissions per country.
    """
    if not os.path.exists(lumiere_dir):
        print("No Lumiere data found. Skipping.")
        print("To add Lumiere data, download Excel files from lumiere.obs.coe.int")
        print(f"and place them in {lumiere_dir}")
        return pd.DataFrame()
    
    files = [f for f in os.listdir(lumiere_dir) if f.endswith(('.xlsx', '.xls', '.csv'))]
    if not files:
        print("No Lumiere files found in data/lumiere/")
        return pd.DataFrame()
    
    dfs = []
    for f in files:
        filepath = os.path.join(lumiere_dir, f)
        try:
            if f.endswith('.csv'):
                df = pd.read_csv(filepath)
            else:
                df = pd.read_excel(filepath)
            dfs.append(df)
            print(f"  Loaded {f}: {len(df)} rows")
        except Exception as e:
            print(f"  ERROR loading {f}: {e}")
    
    if dfs:
        df_lumiere = pd.concat(dfs, ignore_index=True)
        df_lumiere = df_lumiere.drop_duplicates()
        print(f"\nTotal Lumiere entries: {len(df_lumiere)}")
        return df_lumiere
    
    return pd.DataFrame()

print("=" * 60)
print("STEP 6 (optional): Loading Lumiere admissions data...")
print("=" * 60)
df_lumiere = load_lumiere_data()

if not df_lumiere.empty:
    df_lumiere.to_csv('data/lumiere_balkan.csv', index=False)
    print("Saved Lumiere data")

STEP 6 (optional): Loading Lumiere admissions data...
  Loaded lumiere_FULL_data.csv: 1297 rows

Total Lumiere entries: 1297
Saved Lumiere data


---
## Summary

### What we built:

| File | Description | Source |
|------|------------|--------|
| `data/balkan_movies_master.csv` | **Main dataset** — one row per film, all sources merged | All |
| `data/wikidata_balkan_films.csv` | Raw Wikidata extract | Wikidata SPARQL |
| `data/imdb_balkan_films.csv` | IMDb-enriched subset | IMDb TSV dumps |
| `data/tmdb_balkan_films.csv` | TMDb-enriched subset | Kaggle/TMDb |
| `data/lumiere_balkan.csv` | Admissions data (if collected) | Lumiere |

### Master dataset columns:

| Column | Description | Source |
|--------|------------|--------|
| `title_final` | Movie title (English preferred) | Wikidata > IMDb > TMDb |
| `year_final` | Release year | Wikidata > IMDb |
| `era` | Historical period (e.g. 'Black Wave', 'Wars Era') | Derived |
| `country` | Country of origin (pipe-separated) | Wikidata |
| `director` / `director_names` | Director(s) | Wikidata + IMDb |
| `genres_final` | Genres (pipe-separated) | IMDb > TMDb > Wikidata |
| `runtime_final` | Runtime in minutes | IMDb > TMDb > Wikidata |
| `imdb_rating` / `imdb_votes` | IMDb user rating & vote count | IMDb |
| `tmdb_rating` / `tmdb_votes` | TMDb user rating & vote count | TMDb |
| `budget_final` | Production budget (USD) | TMDb > Wikidata |
| `revenue_final` | Box office revenue (USD) | TMDb > Wikidata |
| `plot_overview` | Plot summary text | TMDb |
| `production_companies` | Production companies | TMDb |
| `award` / `num_awards` | Awards received | Wikidata |
| `original_language` | Original language code | TMDb |

### Next step: 
Open `02_analysis.ipynb` to visualize and analyze this data!